In [ ]:
import os
import nibabel as nib
import torch
import torchvision
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torch.nn.utils.parametrize as parametrize
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid
from torch.utils.data.dataloader import DataLoader
from torch.utils.data import random_split
%matplotlib inline

# Use a white background for matplotlib figures
matplotlib.rcParams['figure.facecolor'] = '#ffffff'

In [ ]:
# torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [ ]:
data_dir = './Dataset'
print(os.listdir(data_dir))

img_dir = [os.path.join(data_dir, x) for x in os.listdir(data_dir)]
imgs = [nib.load(img_dir[i]) for i in [0,1,2,5]]
imgs_data = [torch.tensor(i.get_fdata()).unsqueeze(0) for i in imgs]

In [ ]:
random_seed = 20220509
torch.manual_seed(random_seed);

In [ ]:
def Data_Normalization(imgs_data):
    max_value = []
    min_value = []
    for img in imgs_data:
        max_value.append(torch.max(img))
        min_value.append(torch.min(img))
    imgs_data = [2*((x-mi)/(ma-mi)-0.5) for x, ma, mi in zip(imgs_data, max_value, min_value)]
    return imgs_data, max_value, min_value

In [ ]:
imgs_data, max_value, min_value = Data_Normalization(imgs_data)

Create my own dataset:

In [ ]:
# # In order to assign 'labels' to data
# import os
# import pandas as pd
# from torchvision.io import read_image

# class CustomImageDataset(Dataset):
#     def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
#         self.img_labels = pd.read_csv(annotations_file)
#         self.img_dir = img_dir
#         self.transform = transform
#         self.target_transform = target_transform

#     def __len__(self):
#         return len(self.img_labels)

#     def __getitem__(self, idx):
#         img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
#         image = read_image(img_path)
#         label = self.img_labels.iloc[idx, 1]
#         if self.transform:
#             image = self.transform(image)
#         if self.target_transform:
#             label = self.target_transform(label)
#         return image, label

In [ ]:
# imgs_to_be_plotted = imgs_data[0][imgs_data[0].bool()]
# hist = torch.histc(imgs_to_be_plotted, bins=2, min=-1, max=1)
# x = range(2)
# plt.bar(x, hist, align='center', color=['forestgreen'])
# plt.xlabel('Bins')
# plt.ylabel('Frequency')
# plt.show()

In [ ]:
# imgs_to_be_plotted = imgs_data[0][imgs_data[0].bool()]
# hist = torch.histc(imgs_to_be_plotted, bins=2000, min=250, max=2200)
# x = range(2000)
# plt.bar(x, hist, align='center', color=['forestgreen'])
# plt.xlabel('Bins')
# plt.ylabel('Frequency')
# plt.show()

In [ ]:
# for s in range(len(imgs_data)):
#     for t in range(imgs_data[s].size(-1)):
#         imgs_data[s][:,:,:,:,t] = T.Normalize(1000, 1000)(imgs_data[s][:,:,:,:,t])

In [ ]:
imgs_data[2].shape

In [ ]:
val_size = 0
train_size = len(imgs) - val_size

# train_ds, val_ds = random_split(imgs_data, [train_size, val_size])
train_ds = imgs_data

Helper functions for using GPU

In [ ]:
def get_default_device():
    """Pick GPU if available, else CPU"""
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')
    
def to_device(data, device):
    """Move tensor(s) to chosen device"""
    if isinstance(data, (list,tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True, dtype=torch.float)

class DeviceDataLoader():
    """Wrap a dataloader to move data to a device"""
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device
        
    def __iter__(self):
        """Yield a batch of data after moving it to device"""
        for b in self.dl: 
            yield to_device(b, self.device)

    def __len__(self):
        """Number of batches"""
        return len(self.dl)

We can now create PyTorch data loaders for training and validation.

In [ ]:
batch_size = 4

train_loader = DataLoader(train_ds, batch_size)
# val_loader = DataLoader(val_ds, batch_size*2, num_workers=0, pin_memory = True)

# move to GPU
device = get_default_device()
print(device)
train_loader = DeviceDataLoader(train_loader, device)
# val_loader = DeviceDataLoader(val_loader, device)

Next is the model

In [ ]:
class FrobeniusNormalization(nn.Module):
    def forward(self, X):
        F_norm = torch.linalg.matrix_norm(X).item()
        return(torch.div(X, F_norm))

In [ ]:
class RecVAEModel(nn.Module):
    def __init__(self, enc_out_dim=100, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.):
        super().__init__()
        
        self.sqr_sig_x = sqr_sig_x
        self.sqr_sig_h = sqr_sig_h
        # Encoder: from input(x) to one of the inputs of the hidden layer (enc_x)
        # input: 1 x 91 x 109 x 91
        self.encoder1 = nn.Sequential(
            nn.Conv3d(1, 4, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(4),
            nn.LeakyReLU(0.2, inplace = True)) # output: 4 x 45 x 54 x 45
        
        self.encoder2 = nn.Sequential(
            nn.Conv3d(4, 8, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(8),
            nn.LeakyReLU(0.2, inplace = True)) # output: 8 x 22 x 27 x 22
        
        self.encoder3 = nn.Sequential(
            nn.Conv3d(8, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(16),
            nn.LeakyReLU(0.2, inplace = True)) # output: 16 x 11 x 13 x 11
        
        self.encoder4 = nn.Sequential(
            nn.Conv3d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(32),
            nn.LeakyReLU(0.2, inplace = True)) # output: 32 x 5 x 6 x 5
        
        self.encoder5 = nn.Sequential(
            nn.Flatten(), 
            nn.Linear(32*5*6*5, enc_out_dim),
            nn.Tanh()) # output: enc_out_dim
        
        
        # Hidden: from (enc_x,h_{t-1}) to h_t
        self.hidden2mu = nn.Linear(enc_out_dim+latent_dim, latent_dim)
        self.hidden2log_var = nn.Linear(enc_out_dim+latent_dim, latent_dim)
        
        # Decoder: from h_t to mu_t
        self.decoder1 = nn.Sequential(
            nn.Linear(latent_dim, 32*5*6*5),
            nn.Unflatten(1, (32, 5, 6, 5)),
            nn.BatchNorm3d(32),
            nn.LeakyReLU(0.2, inplace = True))
          
        self.decoder2 = nn.Sequential(
            nn.ConvTranspose3d(32, 16, kernel_size=4, stride=2, padding=1, output_padding=1, bias=False), 
            nn.BatchNorm3d(16),
            nn.LeakyReLU(0.2, inplace = True))
        
        self.decoder3 = nn.Sequential(
            nn.ConvTranspose3d(16, 8, kernel_size=4, stride=2, padding=1, output_padding=(0,1,0), bias=False), 
            nn.BatchNorm3d(8),
            nn.LeakyReLU(0.2, inplace = True))
        
        self.decoder4 = nn.Sequential(
            nn.ConvTranspose3d(8, 4, kernel_size=4, stride=2, padding=1, output_padding=(1,0,1), bias=False), 
            nn.BatchNorm3d(4),
            nn.LeakyReLU(0.2, inplace = True))
        
        self.decoder5 = nn.Sequential(
            nn.ConvTranspose3d(4, 1, kernel_size=4, stride=2, padding=1, output_padding=1, bias=False), 
            nn.Tanh())
                   
            
#         self.decoder1 = nn.Sequential(
#             nn.Linear(latent_dim, 200),
#             nn.ReLU(True))
        
#         self.decoder2 = nn.Sequential(
#             nn.Linear(200, 200),
#             nn.ReLU(True))
        
#         self.decoder3 = nn.Sequential(
#             nn.Linear(200, 200),
#             nn.ReLU(True))
        
#         self.decoder4 = nn.Sequential(
#             nn.Linear(200, 1*91*109*91),
#             nn.Tanh(),
#             nn.Unflatten(1, (1, 91, 109, 91))) # output: 1 x 91 x 109 x 91
        
        
        # g(h_{t-1})
        self.g_transform1 = nn.Linear(latent_dim, 128)
        parametrize.register_parametrization(self.g_transform1, "weight", FrobeniusNormalization())
        
        self.g_transform2 = nn.Linear(128, 128)
        parametrize.register_parametrization(self.g_transform2, "weight", FrobeniusNormalization())
        
        self.g_transform3 = nn.Linear(128, latent_dim)
        parametrize.register_parametrization(self.g_transform3, "weight", FrobeniusNormalization())
        
       
    def g_transform(self, h_old):
        h_new = self.g_transform1(h_old)
        h_new = F.relu(h_new)
        h_new = self.g_transform2(h_new)
        h_new = F.relu(h_new)
        h_new = self.g_transform3(h_new)
        return h_new
     
    
    def encode(self, x):
        enc_x = self.encoder1(x)
        enc_x = self.encoder2(enc_x)
        enc_x = self.encoder3(enc_x)
        enc_x = self.encoder4(enc_x)
        enc_x = self.encoder5(enc_x)
        return enc_x
    
    
    def decode(self, h):
        dec_h = self.decoder1(h)
        dec_h = self.decoder2(dec_h)
        dec_h = self.decoder3(dec_h)
        dec_h = self.decoder4(dec_h)
        dec_h = self.decoder5(dec_h)
        return dec_h
        
    
    def reparametrize(self, mu_h,log_var_h):
        # Reparametrization Trick to allow gradients to backpropagate from the stochastic part of the model
        sigma_h = torch.exp(log_var_h / 2)
        z = torch.randn(size = (mu_h.size(0),mu_h.size(1)))
        z = z.type_as(mu_h) # Setting z to be .cuda when using GPU training
        return mu_h + sigma_h*z
      
    
    def VAE(self, x, h):   
        # encode x and h to get the mu and variance parameters for the latent space
        enc_x = self.encode(x)
        combined = torch.cat((enc_x, h), 1)
        mu_h, log_var_h = self.hidden2mu(combined), self.hidden2log_var(combined)
        
        # sample h
        h = self.reparametrize(mu_h, log_var_h)
        
        # decode  
        mu = self.decode(h)
        return mu, h
    
    
    def training_step(self, batch, h):
        '''h is h_0. It is of size batch_size*latent_dim'''
        batch = batch.to(torch.float32)
        x_list, mu_history, h_history, gh_history = self(batch, h)
        
        temp = 2 * batch_size * len(h_history)
        # calculate loss
        loss1 = sum([torch.sum(torch.pow(x-mu, 2)) for x, mu in zip(x_list, mu_history)])
        loss1 = loss1 / self.sqr_sig_x / temp
        #loss1 = sum([F.mse_loss(x, mu) for x, mu in zip(x_list, mu_history)])
#         loss2 = sum([torch.sum(torch.pow(h-gh, 2)) for h, gh in zip(h_history, gh_history)])
#         loss2 = loss2 / self.sqr_sig_h / temp
        loss2 = 0
#         for h, gh in zip(h_history, gh_history):
#             print(h, gh)
#             #print(torch.isnan(h).any(), torch.isnan(gh).any())
        #loss2 = sum([F.mse_loss(h, gh) for h, gh in zip(h_history, gh_history)])
        #print(loss1, loss2)
        loss = loss1 + loss2
        
        return loss1, loss2, loss
    
    
    def forward(self, x, h):
        tol_time = x.size(-1) # x is of size batch_size*channel*x1*x2*x3*tol_time
        x_list = [x[:,:,:,:,:,t] for t in range(tol_time)]
        h_history = []
        gh_history = []
        mu_history = []
        for t in range(tol_time):
            #print(t)
            gh_history.append(self.g_transform(h))
            mu, h = self.VAE(x_list[t], h)
            h_history.append(h)
            mu_history.append(mu)
        # print("h: ", h_history[50], "\n", "gh: ", gh_history[50])
        return x_list, mu_history, h_history, gh_history
    

model = RecVAEModel()
model = to_device(model, device)

In [ ]:
def fit(epochs, lr, h0, model, train_loader=train_loader, val_loader=val_loader, opt_func=torch.optim.SGD):
    train_loss_history = []
    optimizer = opt_func(model.parameters(), lr)
    for epoch in range(epochs):
        # Training Phase 
        model.train()
        for batch in train_loader:
            loss1, loss2, loss = model.training_step(batch, h0.expand(batch.size(0), -1))
            #print(loss)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        # train_loss_history.append(loss)
        # print("Epoch [{}], train_loss: {:.2f}".format(epoch, loss))
        print("Epoch [{}], train_loss: {:.2f} with loss1: {:.2f} and loss2: {:.2f}".format(epoch, loss, loss1, loss2))
    return train_loss_history

In [ ]:
h0 = torch.rand(1, 50)
h0 = to_device(h0, device)
h0

New decoder

In [ ]:
# enc_out_dim=200, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.
# new model
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history = fit(epochs=500, lr=1e-6, h0=h0, model=model)

In [ ]:
# continued
# enc_out_dim=200, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.
# new model
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history = fit(epochs=500, lr=5e-5, h0=h0, model=model)

In [ ]:
# enc_out_dim=100, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.
# new model
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history = fit(epochs=1000, lr=5e-6, h0=h0, model=model)

In [ ]:
# enc_out_dim=100, latent_dim=100, sqr_sig_x=1., sqr_sig_h=1.
# new model
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history = fit(epochs=1000, lr=5e-6, h0=h0, model=model)

In [ ]:
# enc_out_dim=100, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.
# original model
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history, mu_history = fit(epochs=5, lr=5e-6, h0=h0, model=model)

In [ ]:
# enc_out_dim=100, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.
# new model
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history, mu_history = fit(epochs=500, lr=5e-6, h0=h0, model=model)

In [ ]:
# enc_out_dim=100, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.
# model version 3 
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history, mu_history = fit(epochs=500, lr=1e-6, h0=h0, model=model)

In [ ]:
# enc_out_dim=100, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.
# model version 3 
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history, mu_history = fit(epochs=500, lr=2e-6, h0=h0, model=model)

In [ ]:
# enc_out_dim=100, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.
# model version 3 
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history, mu_history = fit(epochs=500, lr=2e-6, h0=h0, model=model)

In [ ]:
# enc_out_dim=100, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.
# model version 3 
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history, mu_history = fit(epochs=500, lr=5e-6, h0=h0, model=model)

In [ ]:
# enc_out_dim=100, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.
# model version 3 
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history, mu_history = fit(epochs=500, lr=5e-6, h0=h0, model=model)

In [ ]:
# enc_out_dim=100, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.
# model version 3 
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history = fit(epochs=500, lr=5e-6, h0=h0, model=model)

In [ ]:
# enc_out_dim=100, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.
# model version 3 
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history = fit(epochs=500, lr=5e-6, h0=h0, model=model)

In [ ]:
# enc_out_dim=100, latent_dim=50, sqr_sig_x=1., sqr_sig_h=1.
# model version 3 
# delete loss2
# dataset: 4 images (IMCL not included)
train_loss_history = fit(epochs=1000, lr=5e-6, h0=h0, model=model)

In [ ]:
@torch.no_grad()
def evaluate(model, x, h0):
    model.eval()
    x = x.unsqueeze(0)
    x = to_device(x, device)
    _, mu_history, _, _ = model(x, h0)
    return mu_history

In [ ]:
train_ds[0].shape

In [ ]:
train_ds[0].unsqueeze(0).shape

In [ ]:
mu_history = evaluate(model, imgs_data[0], h0)

In [ ]:
len(mu_history)

In [ ]:
mu_history[60].shape

In [ ]:
t_tocompare = 60 # the timestamp on which we are to do comparison

def save_image(mu_history, name, which_one, t=t_tocompare):
    maxim = torch.Tensor.cpu(max_value[which_one]).detach().numpy()
    minim = torch.Tensor.cpu(min_value[which_one]).detach().numpy()
    one_mu = torch.Tensor.cpu(mu_history[t]).detach().numpy()
    one_mu = one_mu[0,:,:,:,:]
    one_mu = (one_mu * .5 + .5) * (maxim - minim) + minim
    img_new = nib.Nifti1Image(one_mu, np.eye(4))
    nib.save(img_new, os.path.join('Generated', name))

In [ ]:
def get_name(file_position, data_dir = './Generated'):
    return os.listdir(data_dir)[file_position]


def show_slices(slices):
   """ Function to display row of image slices """
   fig, axes = plt.subplots(1, len(slices))
   for i, slice in enumerate(slices):
       axes[i].imshow(slice.T, cmap="gray", origin="lower")


def get_plot(file_name, data_dir = './Generated', plot_name = 'Slices', x1=40, x2=40, x3=40, t=t_tocompare):
    img = nib.load(os.path.join(data_dir, file_name))
    img_data = img.get_fdata()
    if data_dir == './Dataset':
        show_slices([img_data[x1,:,:,t], img_data[:,x2,:,t], img_data[:,:,x3,t]])
        plt.suptitle(plot_name)
    else:
        show_slices([img_data[0,x1,:,:], img_data[0,:,x2,:], img_data[0,:,:,x3]])
        plt.suptitle(plot_name) 

In [ ]:
save_image(mu_history=mu_history, name='newmodel_img0_2000epochs.nii.gz', which_one=0)

In [ ]:
get_plot('try0.nii.gz')
get_plot('try1.nii.gz')
get_plot('try2.nii.gz')
get_plot('try3.nii.gz')

In [ ]:
get_plot('newmodeltry0.nii.gz')
get_plot('newmodeltry0_1000epochs.nii.gz')
get_plot('newmodeltry0_1500epochs.nii.gz')
get_plot('newmodeltry0_2000epochs.nii.gz')

In [ ]:
get_plot('newmodeltry0_2000epochs.nii.gz')
get_plot('newmodeltry1_2000epochs.nii.gz')
get_plot('newmodeltry2_2000epochs.nii.gz')
get_plot('newmodeltry3_2000epochs.nii.gz')
get_plot('newmodeltry3_2500epochs.nii.gz')

In [ ]:
get_plot('newmodel_img0_2000epochs.nii.gz', plot_name = 'Generated')
get_plot('I269254_I989324imagedata.nii.gz', './Dataset', 'Original')

In [ ]:
print(os.listdir(data_dir))

In [ ]:
max_value

The following can be ignored

In [ ]:
h_new = torch.rand(1, 10)
h_new = to_device(h_new, device)
h_new

In [ ]:
h_new = torch.tensor([[1.1949, -0.6692,  0.2503,  0.0041, -2.3825,  0.6658, -0.0488, -0.7783,
          0.4877,  0.7172]])
h_new = to_device(h_new, device)
h_new

In [ ]:
dec_h_new = model.decode(h_new)
dec_h_new = dec_h_new.squeeze(0)
dec_h_new.shape

In [ ]:
dec_h_new = torch.Tensor.cpu(dec_h_new)
dec_h_new = dec_h_new.detach().numpy()
dec_h_new.shape

In [ ]:
dec_h_new = dec_h_new * 1000 + 1000

In [ ]:
img_new = nib.Nifti1Image(dec_h_new, np.eye(4))
nib.save(img_new, os.path.join('Generated', 'firstone.nii.gz'))  

Run one by one:

In [ ]:
epochs=0
lr=1e-5
opt_func=torch.optim.SGD
train_loss_history = []
optimizer = opt_func(model.parameters(), lr)

In [ ]:
for batch in train_loader:
    debug_batch = batch
    break
debug_batch.shape

In [ ]:
debug_batch = debug_batch.to(torch.float32)

In [ ]:
x = debug_batch
tol_time = x.size(-1) # x is of size batch_size*channel*x1*x2*x3*tol_time
x_list = [x[:,:,:,:,:,t] for t in range(tol_time)]
h_history = []
gh_history = []
mu_history = []
t=0
h = h0.expand(batch.size(0), -1)

In [ ]:
h

In [ ]:
gh_history.append(model.g_transform(h))
gh_history